# Grid as a Weighted Directed Network

Bidding zones are nodes, cross-border interconnectors are edges, and physical flow is the edge weight &ndash; the grid as a network rather than a collection of individual border leaderboards.

This article asks three questions: which zones are structurally important once you account for *all* their connections, not just their single busiest border (centrality); do the clusters that physical flow naturally forms line up with the market-coupling regions that actually clear prices (community detection vs CCR mechanism); and how much of the network's integrity rests on a handful of zones or borders (robustness).

In [ ]:
# common.py is shared across every article and changes often; without this a
# kernel that already imported it keeps serving the stale module.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import networkx as nx
import pandas as pd
import plotly.express as px
from common import (
    CCR_MECHANISM_COLORS,
    border_ccr_mechanism,
    drop_ukraine,
    fetch_flows_range,
    network_graph_chart,
    save_chart,
)

## Pulling a year of flow data

Same window as the self-organized-criticality article: a full year ending 10 June 2026, the day Core's Advanced Hybrid Coupling (AHC) went live. Keeping the whole sample inside one stable market-structure regime matters more here than for the SOC piece &ndash; a network built from flows that straddle a capacity-allocation reform would mix two different topologies into one snapshot.

In [ ]:
date_from = "2025-06-10"
date_to = "2026-06-10"

flows = fetch_flows_range(date_from, date_to)
print(f"{len(flows):,} observations, {flows['border'].nunique()} borders, {date_from} to {date_to}")
flows.head()

## Building the flow network

Each border becomes up to two directed edges, not one: `net_flow_mw` is signed (positive one way, negative the other), and the `bidding_zone_from`/`bidding_zone_to` columns are a fixed nominal pair per border, not the actual flow direction. Summing the positive and negative parts separately gives the real directed throughput each way &ndash; collapsing on the nominal column direction instead would silently throw away every border that mostly flows "backward," which turned out to zero out *every* cycle in the network during scoping.

UA_IPS borders are dropped, same convention as the congestion and SOC articles: wartime grid conditions and emergency EU synchronisation drive those numbers, not ordinary market behaviour.

In [ ]:
def build_flow_graph(flows: pd.DataFrame) -> nx.DiGraph:
    flows = drop_ukraine(flows)
    G = nx.DiGraph()
    for (zone_from, zone_to), sub in flows.groupby(["bidding_zone_from", "bidding_zone_to"]):
        forward = sub.loc[sub["net_flow_mw"] > 0, "net_flow_mw"].sum()
        backward = -sub.loc[sub["net_flow_mw"] < 0, "net_flow_mw"].sum()
        if forward > 0:
            G.add_edge(zone_from, zone_to, weight=forward)
        if backward > 0:
            G.add_edge(zone_to, zone_from, weight=backward)
    return G


G = build_flow_graph(flows)
print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"strongly connected: {nx.is_strongly_connected(G)}")

## Centrality

Three measures, three different questions:

- **PageRank** (weighted, directed) &ndash; which zones are structurally important once you weight by *who* flows into them, not just how much. Preferred over plain eigenvector centrality here: eigenvector centrality can degenerate to all-zero on a directed graph that isn't strongly connected, and while this window's graph checks out as strongly connected, PageRank is the robust default that doesn't depend on that holding for every future window.
- **In/out-strength** (weighted degree) &ndash; the simple baseline, total MW flowing in and out of each zone.
- **Betweenness** (weighted by inverse flow, since betweenness needs distance/cost semantics and raw flow is a strength, not a cost) &ndash; which zones sit on the most paths between other zone pairs, i.e. transit hubs rather than high-volume endpoints.

In [ ]:
pagerank = nx.pagerank(G, weight="weight")

G_dist = G.copy()
for _, _, d in G_dist.edges(data=True):
    d["distance"] = 1 / d["weight"]
betweenness = nx.betweenness_centrality(G_dist, weight="distance")

centrality = pd.DataFrame(
    {
        "pagerank": pagerank,
        "betweenness": betweenness,
        "strength_in_mw": dict(G.in_degree(weight="weight")),
        "strength_out_mw": dict(G.out_degree(weight="weight")),
    }
).sort_values("pagerank", ascending=False)
centrality.head(10)

In [ ]:
fig = px.bar(
    centrality.reset_index(names="zone").sort_values("pagerank").tail(10),
    x="pagerank",
    y="zone",
    orientation="h",
    title=f"Top 10 zones by PageRank, {date_from} to {date_to}",
)
fig.show()
save_chart(fig, "pagerank_top10")

In [ ]:
fig = px.bar(
    centrality.reset_index(names="zone").sort_values("betweenness").tail(10),
    x="betweenness",
    y="zone",
    orientation="h",
    title=f"Top 10 zones by betweenness centrality (transit hubs), {date_from} to {date_to}",
)
fig.show()
save_chart(fig, "betweenness_top10")

## Community detection vs market-coupling regions

Louvain community detection on the flow-weighted graph (direction collapsed, both-way weights summed &ndash; Louvain's modularity optimization is defined on undirected graphs) finds clusters from physical flow alone, with no knowledge of market structure. Comparing that against `border_ccr_mechanism()` &ndash; the real CCR capacity-allocation mechanism per border (Core flow-based, Nordic/Hansa/Baltic/SEE/SWE NTC, Switzerland explicit) &ndash; checks whether physical clustering actually lines up with how prices get coupled, or whether the two cut across each other.

Each zone gets a single CCR label: the mechanism of its highest-weighted incident border, since a zone can straddle more than one CCR.

In [ ]:
G_undirected = nx.Graph()
for u, v, d in G.edges(data=True):
    w = d["weight"]
    if G_undirected.has_edge(u, v):
        G_undirected[u][v]["weight"] += w
    else:
        G_undirected.add_edge(u, v, weight=w)

communities = nx.algorithms.community.louvain_communities(G_undirected, weight="weight", seed=42)
zone_community = {zone: f"community {i}" for i, comm in enumerate(communities) for zone in comm}
print(f"{len(communities)} communities: " + ", ".join(f"{{{', '.join(sorted(c))}}}" for c in communities))

In [ ]:
zone_mechanism_weight: dict[str, dict[str, float]] = {}
for u, v, d in G.edges(data=True):
    mechanism = border_ccr_mechanism(f"{u} -> {v}")
    for zone in (u, v):
        zone_mechanism_weight.setdefault(zone, {})
        zone_mechanism_weight[zone][mechanism] = zone_mechanism_weight[zone].get(mechanism, 0) + d["weight"]

zone_mechanism = {zone: max(weights, key=weights.get) for zone, weights in zone_mechanism_weight.items()}

comparison = pd.DataFrame(
    {
        "zone": list(G.nodes()),
        "community": [zone_community[z] for z in G.nodes()],
        "mechanism": [zone_mechanism[z] for z in G.nodes()],
    }
)
crosstab = pd.crosstab(comparison["community"], comparison["mechanism"])
purity = crosstab.max(axis=1).sum() / crosstab.sum().sum()
print(
    f"community-mechanism purity: {purity:.1%} (zones whose community's dominant mechanism matches their own)"
)
crosstab

In [ ]:
fig = network_graph_chart(
    G,
    groups=zone_community,
    color_map=None,
    title=f"Flow network, colored by Louvain community, {date_from} to {date_to}",
)
fig.show()
save_chart(fig, "network_by_community")

In [ ]:
fig = network_graph_chart(
    G,
    groups=zone_mechanism,
    color_map=CCR_MECHANISM_COLORS,
    title=f"Flow network, colored by CCR capacity-allocation mechanism, {date_from} to {date_to}",
)
fig.show()
save_chart(fig, "network_by_ccr_mechanism")

## Robustness: random failure vs targeted attack

Removing nodes (zones) or edges (borders) one at a time and tracking how much of the network stays connected &ndash; measured as the largest weakly-connected component's share of all zones &ndash; separates two different failure modes. Removing in **PageRank/betweenness rank order** simulates a targeted attack or a worst-case cascading failure starting at the most important zone or border; removing in **random order**, averaged over many trials, simulates ordinary uncorrelated outages. The gap between the two curves, not either curve alone, is the finding: real networks are typically far more resilient to random failure than to targeted attack, and the size of that gap is a measure of how concentrated the network's integrity is in a few zones/borders.

In [ ]:
import random


def giant_component_fraction(G: nx.DiGraph, n_total: int) -> float:
    if G.number_of_nodes() == 0:
        return 0.0
    return max(len(c) for c in nx.weakly_connected_components(G)) / n_total


def node_robustness_curve(G: nx.DiGraph, order: list[str]) -> list[float]:
    G = G.copy()
    n_total = G.number_of_nodes()
    sizes = [giant_component_fraction(G, n_total)]
    for node in order:
        G.remove_node(node)
        sizes.append(giant_component_fraction(G, n_total))
    return sizes


n_random_trials = 50
targeted_order = list(centrality.sort_values("pagerank", ascending=False).index)

targeted = node_robustness_curve(G, targeted_order)
random_trials = [
    node_robustness_curve(G, random.sample(list(G.nodes()), G.number_of_nodes()))
    for _ in range(n_random_trials)
]
random_mean = pd.DataFrame(random_trials).mean().tolist()

node_robustness = pd.DataFrame(
    {
        "zones_removed": range(len(targeted)),
        "targeted (by PageRank)": targeted,
        "random (mean of 50)": random_mean,
    }
).set_index("zones_removed")

fig = px.line(
    node_robustness,
    title=f"Giant component size vs zones removed, targeted vs random, {date_from} to {date_to}",
    labels={"value": "Giant component (fraction of zones)", "zones_removed": "Zones removed"},
)
fig.show()
save_chart(fig, "node_robustness")

In [ ]:
def halfway_point(curve: list[float]) -> int:
    return next((i for i, v in enumerate(curve) if v < 0.5), len(curve))


t_zones, r_zones = halfway_point(targeted), halfway_point(random_mean)
trials = n_random_trials
print(f"targeted: giant component drops below 50% after removing {t_zones} zones")
print(f"random:   giant component drops below 50% after removing {r_zones} zones (mean of {trials} trials)")

In [ ]:
def edge_robustness_curve(G: nx.DiGraph, order: list[tuple[str, str]]) -> list[float]:
    G = G.copy()
    n_total = G.number_of_nodes()
    sizes = [giant_component_fraction(G, n_total)]
    for edge in order:
        G.remove_edge(*edge)
        sizes.append(giant_component_fraction(G, n_total))
    return sizes


edge_betweenness = nx.edge_betweenness_centrality(G_dist, weight="distance")
targeted_edge_order = sorted(edge_betweenness, key=edge_betweenness.get, reverse=True)

targeted_edges = edge_robustness_curve(G, targeted_edge_order)
random_edge_trials = [
    edge_robustness_curve(G, random.sample(list(G.edges()), G.number_of_edges()))
    for _ in range(n_random_trials)
]
random_edge_mean = pd.DataFrame(random_edge_trials).mean().tolist()

edge_robustness = pd.DataFrame(
    {
        "borders_removed": range(len(targeted_edges)),
        "targeted (by betweenness)": targeted_edges,
        "random (mean of 50)": random_edge_mean,
    }
).set_index("borders_removed")

fig = px.line(
    edge_robustness,
    title=f"Giant component size vs borders removed, targeted vs random, {date_from} to {date_to}",
    labels={"value": "Giant component (fraction of zones)", "borders_removed": "Borders removed"},
)
fig.show()
save_chart(fig, "edge_robustness")

In [ ]:
t_edges, r_edges = halfway_point(targeted_edges), halfway_point(random_edge_mean)
trials = n_random_trials
print(f"targeted: giant component drops below 50% after removing {t_edges} borders")
print(f"random:   giant component drops below 50% after removing {r_edges} borders (mean of {trials} trials)")

## Phase 1 findings

**Centrality splits into two different kinds of "important."** PageRank ranks `IT_CNOR`, `IT_CSUD`, `HU`, `RO`, `IT_NORD` highest &ndash; these are high-*volume* zones, mostly Italy's internal zone-to-zone flows (Italy is split into 7 bidding zones, so a lot of its domestic transfer shows up as "cross-border" flow in this dataset). Betweenness tells a different story: `DE_LU` (0.63), `HU` (0.45), and `AT` (0.40) top that ranking &ndash; these are *transit* zones, sitting on the paths between other zone pairs rather than being high-volume endpoints themselves. Germany-Luxembourg is the clearest case: unremarkable PageRank, by far the highest betweenness &ndash; it's Europe's crossroads, not its biggest mover.

**Louvain communities only partly line up with real market coupling.** 6 communities emerge from physical flow alone, matching geography reasonably well (a Balkans/SEE cluster, a Nordic-Baltic cluster, a southern-Italy-and-islands cluster, a Norway-only cluster, and two mixed Western/Central clusters), but purity against the actual CCR capacity-allocation mechanism (`border_ccr_mechanism`) is only **65.1%** &ndash; roughly a third of zones sit in a physical-flow community whose dominant market-coupling mechanism isn't their own. That gap is the actual finding: physical flow clusters don't fully respect the regulatory/market boundaries that clear prices, which is exactly the kind of mismatch #1's original framing was looking for (see Phase 2 below for digging into *which* zones and *why*).

**Robustness shows almost no gap between targeted and random failure.** Removing zones by PageRank rank drops the giant component below 50% after 13 removals; random removal (averaged over 50 trials) takes 14. Removing borders by betweenness takes 120; random removal takes 117. Both pairs are close enough to call "no meaningful asymmetry" at this granularity &ndash; a departure from the classic scale-free "robust to random failure, fragile to targeted attack" signature, and worth taking at face value rather than forcing the expected story: at the continent-wide bidding-zone level, this network's integrity doesn't appear concentrated in a small set of critical nodes or borders the way a hub-and-spoke or scale-free topology would predict. That's consistent with the centrality split above &ndash; Europe's grid has distinct volume hubs (Italy's internal zones) and distinct transit hubs (`DE_LU`, `HU`, `AT`), but no single zone or border that is simultaneously both, which is likely why no small targeted set breaks the network dramatically faster than chance.

**Phase 2 (deferred):** dig into *which* zones drive the 65.1%-purity mismatch and why &ndash; likely candidates are zones whose community membership is dominated by a neighbor in a different CCR (e.g. `NO_2`/`DK_1` landing in the same community as `AT`/`CZ`/`DE_LU`/`NL` despite being Nordic/Hansa-mechanism zones) &ndash; and triangulate against `ZONE_REGIONS` (geographic buckets) as a second ground truth to see whether the mismatch tracks geography, market mechanism, or neither.